# Module 04 — Universal Company AI Assistant
**VS Code/local version: Parts A–F**

Same beginner-friendly project as Colab, but with a proper local virtual environment and `.env` file.

## Part A — Local setup
Run in VS Code terminal:
```bash
mkdir universal-ai-assistant
cd universal-ai-assistant
python -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip
pip install langchain langchain-groq python-dotenv pandas
```
Create `.env` with `GROQ_API_KEY=your_key_here`. Also add `.env` and `.venv/` to `.gitignore`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in .env")
print("Groq API key loaded.")

In [ ]:
from langchain_groq import ChatGroq
llm_test = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)
print(llm_test.invoke("Say: Groq connection is working.").content)

## Part B — Company data and normal Python functions
A LangChain tool starts as a normal Python function. Keep company data simple so students can customize it easily.

In [ ]:
COMPANY_CONFIG = {
    "company_name": "TechNova Solutions",
    "location": "Karachi, Pakistan",
    "working_hours": "Monday to Friday, 9 AM to 6 PM",
    "contact_email": "hello@technova.example",
    "phone": "+92-300-0000000",
    "about": "We build practical AI, web, mobile and automation solutions.",
    "services": {
        "website": {"display_name": "Business Website", "starting_price_usd": 1500},
        "chatbot": {"display_name": "AI Website Chatbot", "starting_price_usd": 2000},
        "mobile app": {"display_name": "Mobile App", "starting_price_usd": 5000},
        "computer vision": {"display_name": "Computer Vision System", "starting_price_usd": 4500},
        "automation": {"display_name": "Business Automation", "starting_price_usd": 2500}
    },
    "faqs": {
        "payment": "40% advance, 40% after first demo and 20% before final delivery.",
        "timeline": "Small projects usually take 2–4 weeks.",
        "support": "30 days of free basic support after delivery."
    }
}

def normal_get_company_name():
    return COMPANY_CONFIG["company_name"]

normal_get_company_name()

## Part C — Business logic and LangChain tools

In [ ]:
from datetime import datetime
from pathlib import Path
import csv, ast, operator
from langchain.tools import tool

LEADS_FILE = Path("leads.csv")

def get_company_information_logic(topic: str = "general") -> str:
    topic = topic.lower().strip()
    c = COMPANY_CONFIG
    if topic in ["general", "about", "company"]:
        return f"{c['company_name']} | {c['about']} | {c['location']} | {c['working_hours']} | {c['contact_email']} | {c['phone']}"
    if topic in ["services", "service"]:
        return "\n".join([f"- {v['display_name']}: starts from ${v['starting_price_usd']}" for v in c['services'].values()])
    if topic in ["location", "address"]: return c["location"]
    if topic in ["hours", "working hours", "timing"]: return c["working_hours"]
    if topic in ["contact", "email", "phone"]: return f"{c['contact_email']} | {c['phone']}"
    if topic in c["faqs"]: return c["faqs"][topic]
    return "That information is not available in official company data."

def normalize_service_name(service: str) -> str:
    service = service.lower().strip()
    aliases = {"web":"website", "site":"website", "ai bot":"chatbot", "bot":"chatbot", "app":"mobile app", "vision":"computer vision", "cv":"computer vision"}
    return aliases.get(service, service)

def get_service_price_logic(service: str) -> str:
    key = normalize_service_name(service)
    services = COMPANY_CONFIG["services"]
    if key not in services:
        return "Service not found. Available: " + ", ".join(services.keys())
    item = services[key]
    return f"{item['display_name']} starts from ${item['starting_price_usd']} USD."

_ALLOWED_OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
    if isinstance(node, ast.BinOp):
        left, right = _safe_eval(node.left), _safe_eval(node.right)
        op = type(node.op)
        if op not in _ALLOWED_OPERATORS: raise ValueError("Operator not allowed")
        return _ALLOWED_OPERATORS[op](left, right)
    if isinstance(node, ast.UnaryOp):
        op = type(node.op)
        if op not in _ALLOWED_OPERATORS: raise ValueError("Operator not allowed")
        return _ALLOWED_OPERATORS[op](_safe_eval(node.operand))
    raise ValueError("Only basic arithmetic is allowed")

def calculator_logic(expression: str) -> str:
    try: return str(round(_safe_eval(ast.parse(expression, mode="eval").body), 2))
    except Exception as e: return f"Calculation error: {e}"

def save_lead_logic(name: str, email: str, requirement: str) -> str:
    name, email, requirement = name.strip(), email.strip(), requirement.strip()
    if not name or not email or not requirement: return "Lead was not saved. Name, email and requirement are required."
    exists = LEADS_FILE.exists()
    with LEADS_FILE.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["timestamp", "name", "email", "requirement"])
        if not exists: writer.writeheader()
        writer.writerow({"timestamp": datetime.now().isoformat(timespec="seconds"), "name": name, "email": email, "requirement": requirement})
    return f"Lead saved successfully for {name}."

@tool
def get_company_information(topic: str = "general") -> str:
    """Get official company information, services, location, hours, contact details or FAQs."""
    return get_company_information_logic(topic)
@tool
def get_service_price(service: str) -> str:
    """Get the official starting price of a company service."""
    return get_service_price_logic(service)
@tool
def calculator(expression: str) -> str:
    """Calculate basic arithmetic safely, including discounts, percentages and totals."""
    return calculator_logic(expression)
@tool
def save_lead(name: str, email: str, requirement: str) -> str:
    """Save an interested customer's name, email and requirement."""
    return save_lead_logic(name, email, requirement)

tools = [get_company_information, get_service_price, calculator, save_lead]
print(get_service_price.invoke({"service": "website"}))
print(calculator.invoke({"expression": "1500-(1500*0.10)"}))

## Part D — Build the assistant

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = f"""You are the official website assistant for {COMPANY_CONFIG['company_name']}.
Use company tools for official information and prices. Never invent company data.
Use calculator for calculations. If a visitor is interested, collect name, email and requirement.
Only say a lead was saved if save_lead succeeds. Keep answers short and professional."""

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2)
assistant = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

def ask(message: str):
    result = assistant.invoke({"messages": [{"role": "user", "content": message}]})
    answer = result["messages"][-1].content
    print(answer)
    return answer

In [ ]:
ask("What services do you provide?")
ask("How much does an AI chatbot cost?")
ask("Give me 15% discount on the chatbot price.")
ask("My name is Usman, usman@example.com, and I need an ecommerce chatbot. Save my lead.")

## Part E — Simple conversation memory

In [ ]:
chat_history = []
def chat(message: str):
    chat_history.append({"role": "user", "content": message})
    result = assistant.invoke({"messages": chat_history})
    answer = result["messages"][-1].content
    chat_history.append({"role": "assistant", "content": answer})
    print(answer)
    return answer

chat("My name is Ahmed.")
chat("What is my name?")

## Part F — Final student project
Customize `COMPANY_CONFIG` for a clinic, school, restaurant, software house, real-estate company, ecommerce store, travel agency, gym or university. Keep at least 4 tools and run 10 tests.

Next: Part G converts this exact logic into Streamlit.

In [ ]:
tests = [
    "What services do you provide?",
    "Where are you located?",
    "How much does a chatbot cost?",
    "Give me 15% discount on the chatbot price.",
    "I have a budget of $3000. What can I build?",
    "What are your working hours?",
    "What is your payment policy?",
    "Can you build computer vision systems?",
    "My name is Sara, sara@example.com, I need business automation. Save my lead.",
    "What information do you not have?"
]
for q in tests:
    print("\nUSER:", q)
    ask(q)